In [ ]:
library(Seurat)
library(MuDataSeurat)
library(CellChat)
library(reticulate)
library(uwot)
library(ComplexHeatmap)
library(wordcloud)

library(future)

# Create Seurat Objects

In [ ]:
# Specify the path of your h5ad file
path <- "/data/projects/manuela/aging_interorgan/annotated-interorgan_sub_new-soupxed.h5ad"

In [ ]:
# Create seurat object
seu <- ReadH5AD(path)

In [ ]:
# Subset to remove 'EC-1(gEC)' cells
seu <- subset(seu, subset = mixed_celltype != 'EC-1(gEC)')

# Drop the 'EC-1(gEC)' level from mixed_celltype
seu@meta.data$mixed_celltype <- droplevels(seu@meta.data$mixed_celltype)

In [ ]:
# Subset the Seurat object based on the condition
seu_ctrl <- seu[,seu@meta.data$condition == 'ctrl' ]
seu_age <- seu[,seu@meta.data$condition == 'age' ]
seu_DR <- seu[,seu@meta.data$condition == 'DR' ]
seu_sAct <- seu[,seu@meta.data$condition == 'sAct' ]
seu_combi <- seu[,seu@meta.data$condition == 'combi' ]

In [ ]:
# Adjust levels
seu_ctrl$celltype<- droplevels(seu_ctrl$celltype)
seu_age$celltype<- droplevels(seu_age$celltype)
seu_DR$celltype<- droplevels(seu_DR$celltype)
seu_sAct$celltype<- droplevels(seu_sAct$celltype)
seu_combi$celltype<- droplevels(seu_combi$celltype)

head(seu_ctrl@meta.data)

In [ ]:
#head(seu@meta.data)
head(seu_age@meta.data$mixed_celltype)
#head(seu@assays$RNA)
#head(seu@assays$counts)

In [ ]:
# List of Seurat objects
seurat_objects <- list(seu_ctrl, seu_age, seu_DR, seu_sAct, seu_combi)

# Loop through each Seurat object
for (seurat_obj in seurat_objects) {
    expected_levels <-  levels(seurat_obj$celltype)
    if (all(expected_levels %in% seurat_obj$celltype)) {
    cat("All expected levels are present in the celltype column of this Seurat object.\n")
    } else {
    cat("Some expected levels are not present in the celltype column of this Seurat object.\n")
  }
}

# Create Cellchat objects

In [ ]:
# 1.Create cellchat objects
cellchat <- createCellChat(object = seu, group.by = "mixed_celltype", assay = "RNA")

cellchat_ctrl <- createCellChat(object = seu_ctrl, group.by = "mixed_celltype", assay = "RNA")
cellchat_age <- createCellChat(object = seu_age, group.by = "mixed_celltype", assay = "RNA")
cellchat_DR <- createCellChat(object = seu_DR, group.by = "mixed_celltype", assay = "RNA")
cellchat_sAct <- createCellChat(object = seu_sAct, group.by = "mixed_celltype", assay = "RNA")
cellchat_combi <- createCellChat(object = seu_combi, group.by = "mixed_celltype", assay = "RNA")

cellchat_objects <- list(
  cellchat, cellchat_ctrl, cellchat_age, cellchat_DR, cellchat_sAct, cellchat_combi
)

In [ ]:
levels(cellchat@idents)
levels(cellchat_ctrl@idents)
levels(cellchat_age@idents)
levels(cellchat_DR@idents)
levels(cellchat_sAct@idents)
levels(cellchat_combi@idents)

# Run cellchat - write RDS files

In [ ]:
# 2. Add Database
CellChatDB <- CellChatDB.mouse
CellChatDB.use <- CellChatDB # simply use the default CellChatDB

# Assign CellChatDB.use to cellchat@DB
cellchat@DB <- CellChatDB.use
  
# 3. Subset data in cellchat
cellchat <- subsetData(cellchat) # This step is necessary even if using the whole database
future::plan("multisession", workers = 4) # do parallel
    
# Identify overexpressed genes
cat("Identify overexpressed genes \n")
cellchat <- identifyOverExpressedGenes(cellchat)
  
# Identify overexpressed interactions
cat("Identify overexpressed interactions \n")
cellchat <- identifyOverExpressedInteractions(cellchat)
  
# 4. project gene expression data onto PPI (Optional: when running it, USER should set `raw.use = FALSE` in the function `computeCommunProb()` in order to use the projected data)
cat("project gene expression data onto PPI \n")
cellchat <- projectData(cellchat, PPI.mouse)
    
#5. Compute the communication probability matrix for the given pathways  based on the expression of ligand-receptor pairs and their cellular locations
cat("Compute the communication probability matrix \n")
cellchat <- computeCommunProb(cellchat, raw.use = FALSE)
    
#6. Filter out the cell-cell communication if there are only few number of cells in certain cell groups -> reduces potential false positive communications
cat("Filter out the cell-cell communication \n")
cellchat <- filterCommunication(cellchat, min.cells = 10)
    
# 7. communication probability for each signaling pathway by summing the communication probabilities of all involved ligand-receptor pairs
cat("communication probability \n")
cellchat <- computeCommunProbPathway(cellchat)
cellchat <- aggregateNet(cellchat)
saveRDS(cellchat, "./cellchat_subset_objects/cellchat_object_sub_imm.rds")

In [ ]:
cellchat <- readRDS("/data/projects/manuela/aging_interorgan/cellchat_subset_objects/cellchat_object_sub_imm.rds")

# 8. Compute the network centrality scores
cat("Compute the network centrality scores \n")
cellchat <- netAnalysis_computeCentrality(cellchat, slot.name = "netP")

# 9. Save the cellchat object to an RDS file
cat("CellChat object saving in RDS format.\n")
saveRDS(cellchat, "./cellchat_subset_objects/cellchat_centr_sub_imm.rds")

# Print message
cat("Commands executed for ctrl \n")

## ctrl

In [ ]:
# 2. Add Database
CellChatDB <- CellChatDB.mouse
CellChatDB.use <- CellChatDB # simply use the default CellChatDB

# Assign CellChatDB.use to cellchat@DB
cellchat_ctrl@DB <- CellChatDB.use
  
# 3. Subset data in cellchat
cellchat_ctrl <- subsetData(cellchat_ctrl) # This step is necessary even if using the whole database
future::plan("multisession", workers = 4) # do parallel
    
# Identify overexpressed genes
cat("Identify overexpressed genes \n")
cellchat_ctrl <- identifyOverExpressedGenes(cellchat_ctrl)
  
# Identify overexpressed interactions
cat("Identify overexpressed interactions \n")
cellchat_ctrl <- identifyOverExpressedInteractions(cellchat_ctrl)
  
# 4. project gene expression data onto PPI (Optional: when running it, USER should set `raw.use = FALSE` in the function `computeCommunProb()` in order to use the projected data)
cat("project gene expression data onto PPI \n")
cellchat_ctrl <- projectData(cellchat_ctrl, PPI.mouse)
    
#5. Compute the communication probability matrix for the given pathways  based on the expression of ligand-receptor pairs and their cellular locations
cat("Compute the communication probability matrix \n")
cellchat_ctrl <- computeCommunProb(cellchat_ctrl, raw.use = FALSE)
    
#6. Filter out the cell-cell communication if there are only few number of cells in certain cell groups -> reduces potential false positive communications
cat("Filter out the cell-cell communication \n")
cellchat_ctrl <- filterCommunication(cellchat_ctrl, min.cells = 10)
    
# 7. communication probability for each signaling pathway by summing the communication probabilities of all involved ligand-receptor pairs
cat("communication probability \n")
cellchat_ctrl <- computeCommunProbPathway(cellchat_ctrl)
cellchat_ctrl <- aggregateNet(cellchat_ctrl)
saveRDS(cellchat_ctrl, "./cellchat_subset_objects/cellchat_ctrl_object_sub_imm.rds")

In [ ]:
cellchat_ctrl <- readRDS("/data/projects/manuela/aging_interorgan/cellchat_subset_objects/cellchat_ctrl_object_sub_imm.rds")

# 8. Compute the network centrality scores
cat("Compute the network centrality scores \n")
cellchat_ctrl <- netAnalysis_computeCentrality(cellchat_ctrl, slot.name = "netP")

# 9. Save the cellchat_ctrl object to an RDS file
cat("CellChat object saving in RDS format.\n")
saveRDS(cellchat_ctrl, "./cellchat_subset_objects/cellchat_ctrl_centr_sub_imm.rds")

# Print message
cat("Commands executed for ctrl \n")

## age

In [ ]:
# 2. Add Database
CellChatDB <- CellChatDB.mouse
CellChatDB.use <- CellChatDB # simply use the default CellChatDB

# Assign CellChatDB.use to cellchat@DB
cellchat_age@DB <- CellChatDB.use
  
# 3. Subset data in cellchat
cellchat_age <- subsetData(cellchat_age) # This step is necessary even if using the whole database
future::plan("multisession", workers = 4) # do parallel
    
# Identify overexpressed genes
cat("Identify overexpressed genes \n")
cellchat_age <- identifyOverExpressedGenes(cellchat_age)
  
# Identify overexpressed interactions
cat("Identify overexpressed interactions \n")
cellchat_age <- identifyOverExpressedInteractions(cellchat_age)
  
# 4. project gene expression data onto PPI (Optional: when running it, USER should set `raw.use = FALSE` in the function `computeCommunProb()` in order to use the projected data)
cat("project gene expression data onto PPI \n")
cellchat_age <- projectData(cellchat_age, PPI.mouse)
    
#5. Compute the communication probability matrix for the given pathways  based on the expression of ligand-receptor pairs and their cellular locations
cat("Compute the communication probability matrix \n")
cellchat_age <- computeCommunProb(cellchat_age, raw.use = FALSE)
    
#6. Filter out the cell-cell communication if there are only few number of cells in certain cell groups -> reduces potential false positive communications
cat("Filter out the cell-cell communication \n")
cellchat_age <- filterCommunication(cellchat_age, min.cells = 10)
    
# 7. communication probability for each signaling pathway by summing the communication probabilities of all involved ligand-receptor pairs
cat("communication probability \n")
cellchat_age <- computeCommunProbPathway(cellchat_age)
cellchat_age <- aggregateNet(cellchat_age)
saveRDS(cellchat_age, "./cellchat_subset_objects/cellchat_age_object_sub_imm.rds")

In [ ]:
cellchat_age <- readRDS("/data/projects/manuela/aging_interorgan/cellchat_subset_objects/cellchat_age_object_sub_imm.rds")

# 8. Compute the network centrality scores
cat("Compute the network centrality scores \n")
cellchat_age <- netAnalysis_computeCentrality(cellchat_age, slot.name = "netP")

# 9. Save the cellchat_age object to an RDS file
cat("CellChat object saving in RDS format.\n")
saveRDS(cellchat_age, "./cellchat_subset_objects/cellchat_age_centr_sub_imm.rds")

# Print message
cat("Commands executed for age \n")

## DR

In [ ]:
# 2. Add Database
CellChatDB <- CellChatDB.mouse
CellChatDB.use <- CellChatDB # simply use the default CellChatDB

# Assign CellChatDB.use to cellchat@DB
cellchat_DR@DB <- CellChatDB.use
  
# 3. Subset data in cellchat
cellchat_DR <- subsetData(cellchat_DR) # This step is necessary even if using the whole database
future::plan("multisession", workers = 4) # do parallel
    
# Identify overexpressed genes
cat("Identify overexpressed genes \n")
cellchat_DR <- identifyOverExpressedGenes(cellchat_DR)
  
# Identify overexpressed interactions
cat("Identify overexpressed interactions \n")
cellchat_DR <- identifyOverExpressedInteractions(cellchat_DR)
  
# 4. project gene expression data onto PPI (Optional: when running it, USER should set `raw.use = FALSE` in the function `computeCommunProb()` in order to use the projected data)
cat("project gene expression data onto PPI \n")
cellchat_DR <- projectData(cellchat_DR, PPI.mouse)
    
#5. Compute the communication probability matrix for the given pathways  based on the expression of ligand-receptor pairs and their cellular locations
cat("Compute the communication probability matrix \n")
cellchat_DR <- computeCommunProb(cellchat_DR, raw.use = FALSE)
    
#6. Filter out the cell-cell communication if there are only few number of cells in certain cell groups -> reduces potential false positive communications
cat("Filter out the cell-cell communication \n")
cellchat_DR <- filterCommunication(cellchat_DR, min.cells = 10)
    
# 7. communication probability for each signaling pathway by summing the communication probabilities of all involved ligand-receptor pairs
cat("communication probability \n")
cellchat_DR <- computeCommunProbPathway(cellchat_DR)
cellchat_DR <- aggregateNet(cellchat_DR)
saveRDS(cellchat_DR, "./cellchat_subset_objects/cellchat_DR_object_sub_imm.rds")

In [ ]:
cellchat_DR <- readRDS("/data/projects/manuela/aging_interorgan/cellchat_subset_objects/cellchat_DR_object_sub_imm.rds")

# 8. Compute the network centrality scores
cat("Compute the network centrality scores \n")
cellchat_DR <- netAnalysis_computeCentrality(cellchat_DR, slot.name = "netP")

# 9. Save the cellchat_DR object to an RDS file
cat("CellChat object saving in RDS format.\n")
saveRDS(cellchat_DR, "./cellchat_subset_objects/cellchat_DR_centr_sub_imm.rds")

# Print message
cat("Commands executed for DR \n")

## sAct

In [ ]:
# 2. Add Database
CellChatDB <- CellChatDB.mouse
CellChatDB.use <- CellChatDB # simply use the default CellChatDB

# Assign CellChatDB.use to cellchat@DB
cellchat_sAct@DB <- CellChatDB.use
  
# 3. Subset data in cellchat
cellchat_sAct <- subsetData(cellchat_sAct) # This step is necessary even if using the whole database
future::plan("multisession", workers = 4) # do parallel
    
# Identify overexpressed genes
cat("Identify overexpressed genes \n")
cellchat_sAct <- identifyOverExpressedGenes(cellchat_sAct)
  
# Identify overexpressed interactions
cat("Identify overexpressed interactions \n")
cellchat_sAct <- identifyOverExpressedInteractions(cellchat_sAct)
  
# 4. project gene expression data onto PPI (Optional: when running it, USER should set `raw.use = FALSE` in the function `computeCommunProb()` in order to use the projected data)
cat("project gene expression data onto PPI \n")
cellchat_sAct <- projectData(cellchat_sAct, PPI.mouse)
    
#5. Compute the communication probability matrix for the given pathways  based on the expression of ligand-receptor pairs and their cellular locations
cat("Compute the communication probability matrix \n")
cellchat_sAct <- computeCommunProb(cellchat_sAct, raw.use = FALSE)
    
#6. Filter out the cell-cell communication if there are only few number of cells in certain cell groups -> reduces potential false positive communications
cat("Filter out the cell-cell communication \n")
cellchat_sAct <- filterCommunication(cellchat_sAct, min.cells = 10)
    
# 7. communication probability for each signaling pathway by summing the communication probabilities of all involved ligand-receptor pairs
cat("communication probability \n")
cellchat_sAct <- computeCommunProbPathway(cellchat_sAct)
cellchat_sAct <- aggregateNet(cellchat_sAct)
saveRDS(cellchat_sAct, "./cellchat_subset_objects/cellchat_sAct_object_sub_imm.rds")

In [ ]:
cellchat_sAct <- readRDS("/data/projects/manuela/aging_interorgan/cellchat_subset_objects/cellchat_sAct_object_sub_imm.rds")

# 8. Compute the network centrality scores
cat("Compute the network centrality scores \n")
cellchat_sAct <- netAnalysis_computeCentrality(cellchat_sAct, slot.name = "netP")

# 9. Save the cellchat_sAct object to an RDS file
cat("CellChat object saving in RDS format.\n")
saveRDS(cellchat_sAct, "./cellchat_subset_objects/cellchat_sAct_centr_sub_imm.rds")

# Print message
cat("Commands executed for sAct \n")

## combi

In [ ]:
# 2. Add Database
CellChatDB <- CellChatDB.mouse
CellChatDB.use <- CellChatDB # simply use the default CellChatDB

# Assign CellChatDB.use to cellchat@DB
cellchat_combi@DB <- CellChatDB.use
  
# 3. Subset data in cellchat
cellchat_combi <- subsetData(cellchat_combi) # This step is necessary even if using the whole database
future::plan("multisession", workers = 4) # do parallel
    
# Identify overexpressed genes
cat("Identify overexpressed genes \n")
cellchat_combi <- identifyOverExpressedGenes(cellchat_combi)
  
# Identify overexpressed interactions
cat("Identify overexpressed interactions \n")
cellchat_combi <- identifyOverExpressedInteractions(cellchat_combi)
  
# 4. project gene expression data onto PPI (Optional: when running it, USER should set `raw.use = FALSE` in the function `computeCommunProb()` in order to use the projected data)
cat("project gene expression data onto PPI \n")
cellchat_combi <- projectData(cellchat_combi, PPI.mouse)
    
#5. Compute the communication probability matrix for the given pathways  based on the expression of ligand-receptor pairs and their cellular locations
cat("Compute the communication probability matrix \n")
cellchat_combi <- computeCommunProb(cellchat_combi, raw.use = FALSE)
    
#6. Filter out the cell-cell communication if there are only few number of cells in certain cell groups -> reduces potential false positive communications
cat("Filter out the cell-cell communication \n")
cellchat_combi <- filterCommunication(cellchat_combi, min.cells = 10)
    
# 7. communication probability for each signaling pathway by summing the communication probabilities of all involved ligand-receptor pairs
cat("communication probability \n")
cellchat_combi <- computeCommunProbPathway(cellchat_combi)
cellchat_combi <- aggregateNet(cellchat_combi)
saveRDS(cellchat_combi, "./cellchat_subset_objects/cellchat_combi_object_sub_imm.rds")

In [ ]:
cellchat_combi <- readRDS("/data/projects/manuela/aging_interorgan/cellchat_subset_objects/cellchat_combi_object_sub_imm.rds")

# 8. Compute the network centrality scores
cat("Compute the network centrality scores \n")
cellchat_combi <- netAnalysis_computeCentrality(cellchat_combi, slot.name = "netP")

# 9. Save the cellchat_combi object to an RDS file
cat("CellChat object saving in RDS format.\n")
saveRDS(cellchat_combi, "./cellchat_subset_objects/cellchat_combi_centr_sub_imm.rds")

# Print message
cat("Commands executed for combi \n")

# Read in RDS objects

In [ ]:
# Load the cellchat objects from RDS file with centrality scores
cellchat.ctrl <- readRDS("/data/projects/manuela/aging_interorgan/cellchat_subset_objects/cellchat_ctrl_centr_sub_imm.rds")
cellchat.age <- readRDS("/data/projects/manuela/aging_interorgan/cellchat_subset_objects/cellchat_age_centr_sub_imm.rds")
cellchat.DR <- readRDS("/data/projects/manuela/aging_interorgan/cellchat_subset_objects/cellchat_DR_centr_sub_imm.rds")
cellchat.sAct <- readRDS("/data/projects/manuela/aging_interorgan/cellchat_subset_objects/cellchat_sAct_centr_sub_imm.rds")
cellchat.combi <- readRDS("/data/projects/manuela/aging_interorgan/cellchat_subset_objects/cellchat_combi_centr_sub_imm.rds")

object.list <- list(ctrl = cellchat.ctrl, age = cellchat.age, DR = cellchat.DR, sAct = cellchat.sAct, combi = cellchat.combi)# Create a named list of cellchat objects

In [ ]:
cellchat_objects <- list(
 # list(object = cellchat, name = "cellchat"),
  list(object = cellchat.ctrl, name = "ctrl"),
  list(object = cellchat.age, name = "aging"),
  list(object = cellchat.DR, name = "sAct"),
  list(object = cellchat.sAct, name = "DR"),
  list(object = cellchat.combi, name = "combi")
)

# Loop through each cellchat object and apply visualization code
for (cellchat_item in cellchat_objects) {
  cellchat_obj <- cellchat_item$object
  cellchat_name <- cellchat_item$name
  groupSize <- as.numeric(table(cellchat_obj@idents)) # number of cells in each cell group
  par(mfrow = c(1,1), xpd=TRUE)
  
  # Visualize number of interactions with title
  netVisual_circle(cellchat_obj@net$count, vertex.weight = groupSize, weight.scale = TRUE, label.edge = FALSE, title.name = paste("Number of interactions -", cellchat_name))
  
  # Visualize interaction weights/strength with title
  par(mfrow = c(1,1), xpd=TRUE)
  netVisual_circle(cellchat_obj@net$weight, vertex.weight = groupSize, weight.scale = TRUE, label.edge = FALSE, title.name = paste("Interaction weights/strength -", cellchat_name))
}

## Part I: Comparison analysis of multiple datasets with slightly different cell type compositions

In [ ]:
# Extract cell type annotations from each object
celltype_annotations <- lapply(cellchat_objects, function(item) {
  cellchat_obj <- item$object
  cellchat_name <- item$name
  celltypes <- unique(cellchat_obj@idents)
  cat("Cell types in", cellchat_name, ":", paste(celltypes, collapse = ", "), "\n")
  return(celltypes)
})

# Compare cell type compositions
for (i in 1:(length(cellchat_objects) - 1)) {
  for (j in (i + 1):length(cellchat_objects)) {
    obj1_name <- cellchat_objects[[i]]$name
    obj2_name <- cellchat_objects[[j]]$name
    
    if (!identical(celltype_annotations[[i]], celltype_annotations[[j]])) {
      cat("Cell type compositions are different between", obj1_name, "and", obj2_name, "\n")
    } else {
      cat("Cell type compositions are the same between", obj1_name, "and", obj2_name, "\n")
    }
  }
}

### Lift up CellChat object

In [ ]:
levels(cellchat.ctrl@idents)
levels(cellchat.age@idents)
levels(cellchat.DR@idents)
levels(cellchat.sAct@idents)
levels(cellchat.combi@idents)

In [ ]:
# Lift up levels of ctrl
group.new <- levels(cellchat.age@idents)
cellchat.ctrl<- liftCellChat(cellchat.ctrl, group.new)

object.list <- list(ctrl = cellchat.ctrl, age = cellchat.age, DR = cellchat.DR, sAct = cellchat.sAct, combi = cellchat.combi)# Create a named list of cellchat objects
cellchat <- mergeCellChat(object.list, add.names = names(object.list))
#cellchat <- mergeCellChat(object.list, add.names = names(object.list), cell.prefix = TRUE)

### Visualize the inferred signaling network using the lifted object

In [ ]:
# Find common pathways to display
all_pathways <- list()

# Iterate through each named object and extract pathways
for (object_name in names(object.list)) {
  cellchat_obj <- object.list[[object_name]]
  pathways <- cellchat_obj@netP$pathways
  all_pathways[[object_name]] <- pathways
}

# Print the list of combined pathways
all_pathways

# Extract the pathways that are present in all objects
common_pathways <- Reduce(intersect, all_pathways)
common_pathways

In [ ]:
# Hierarchy plot
pathways.show <- c("ANGPT") 
weight.max <- getMaxWeight(object.list, slot.name = c("netP"), attribute = pathways.show) # control the edge weights across different datasets
vertex.receiver = seq(1,10) # Left portion of hierarchy plot the shows signaling to dermal cells and right portion shows signaling to epidermal cells
par(mfrow = c(1,2), xpd=TRUE)
for (i in 1:length(object.list)) {
  netVisual_aggregate(object.list[[i]], signaling = pathways.show, vertex.receiver = vertex.receiver, edge.weight.max = weight.max[1], edge.width.max = 10, signaling.name = paste(pathways.show, names(object.list)[i]))
}

## Subset cellchat obj to desired celltypes

In [ ]:
levels(cellchat@meta[['supertype']])
levels(cellchat@meta[['subtype']])
levels(cellchat@meta[['celltype']])
levels(cellchat@meta[['mixed_celltype']])

In [ ]:
# Define the cell types you want to include
desired_cell_types <- c("Neuromuscular junction nuclei", "Myonuclei", "Endothelial cells")

# Subset the cellchat object based on desired cell types
sub_cellchat <- subsetCellChat(cellchat, idents.use = desired_cell_types)

## Part I: Predict general principles of cell-cell communication

In [ ]:
# Compare the total number of interactions and interaction strength
plot_title <- "Interactions of all celltypes"

gg1 <- compareInteractions(cellchat, show.legend = F, group = c(1,2,3,4,5), title.name = plot_title)
gg2 <- compareInteractions(cellchat, show.legend = F, group = c(1,2,3,4,5), measure = "weight", title.name = plot_title)
gg1 + gg2

In [ ]:
# Compare the total number of interactions and interaction strength
plot_title <- "Interactions of subsettet celltypes"

gg1 <- compareInteractions(sub_cellchat, show.legend = F, group = c(1,2,3,4,5), title.name = plot_title)
gg2 <- compareInteractions(sub_cellchat, show.legend = F, group = c(1,2,3,4,5), measure = "weight", title.name = plot_title)
gg1 + gg2

...

In [ ]:
# Create a vector of sample names to compare with 'ctrl'
samples_to_compare <- c('age', 'DR','sAct', 'combi')

# Initialize an empty list to store the heatmaps
heatmap_list <- list()

# Loop through each sample to compare with 'ctrl'
for (sample in samples_to_compare) {
  # Create the comparison vector
  comparison_vector <- c('ctrl', sample)
  
  # Generate the heatmaps
    gg1 <- netVisual_heatmap(cellchat, comparison = comparison_vector, title.name = paste("Nb interactions:", comparison_vector[1], "vs", comparison_vector[2]))
    gg2 <- netVisual_heatmap(cellchat, comparison = comparison_vector, measure = "weight", title.name = paste("Interaction strength:", comparison_vector[1], "vs", comparison_vector[2]))
    print(gg1 + gg2)
    # Define the file path for saving the heatmap (change the format to 'png' or 'pdf' as needed)
    file_path <- file.path("./cellchat_figures", paste0("heatmap_", sample, ".pdf"))
}

In [ ]:
# Create a vector of sample names to compare with 'ctrl'
samples_to_compare <- c('ctrl', 'DR','sAct', 'combi')

# Initialize an empty list to store the heatmaps
heatmap_list <- list()

# Loop through each sample to compare with 'ctrl'
for (sample in samples_to_compare) {
  # Create the comparison vector
  comparison_vector <- c('age', sample)
  
  # Generate the heatmaps
    gg1 <- netVisual_heatmap(cellchat, comparison = comparison_vector, title.name = paste("Nb interactions:", comparison_vector[1], "vs", comparison_vector[2]))
    gg2 <- netVisual_heatmap(cellchat, comparison = comparison_vector, measure = "weight", title.name = paste("Interaction strength:", comparison_vector[1], "vs", comparison_vector[2]))
    print(gg1 + gg2)
    
    #Generate barplots
   # gg3 <- netVisual_barplot(cellchat, comparison = comparison_vector, title.name = paste("Comparison:", comparison_vector[1], "vs", comparison_vector[2]))
  #  gg4 <- netVisual_barplot(cellchat, comparison = comparison_vector, measure = "weight", title.name = paste("Comparison:", comparison_vector[1], "vs", comparison_vector[2]))
   # print(gg3)
    # Define the file path for saving the heatmap (change the format to 'png' or 'pdf' as needed)
    file_path <- file.path("./cellchat_figures", paste0("heatmap_", sample, ".pdf"))
}

...

### Compare the major sources and targets in 2D space

'Comparing the outgoing and incoming interaction strength in 2D space allows ready identification of the cell populations with significant changes in sending or receiving signals between different datasets.

In [ ]:
cellchat.ctrl <- netAnalysis_computeCentrality(cellchat.ctrl, slot.name = "netP")
cellchat.age <- netAnalysis_computeCentrality(cellchat.age, slot.name = "netP")
cellchat.DR <- netAnalysis_computeCentrality(cellchat.DR, slot.name = "netP")
cellchat.sAct <- netAnalysis_computeCentrality(cellchat.sAct, slot.name = "netP")
cellchat.combi <- netAnalysis_computeCentrality(cellchat.combi, slot.name = "netP")

In [ ]:
num.link <- sapply(object.list, function(x) {rowSums(x@net$count) + colSums(x@net$count)-diag(x@net$count)})
weight.MinMax <- c(min(num.link), max(num.link)) # control the dot size in the different datasets
gg <- list()
for (i in 1:length(object.list)) {
  #gg[[i]] <- netAnalysis_signalingRole_scatter(object.list[[i]], title = names(object.list)[i], weight.MinMax = weight.MinMax)
    #version with scaled achses
    gg[[i]] <- netAnalysis_signalingRole_scatter(object.list[[i]],  title = names(object.list)[i],  weight.MinMax = weight.MinMax) +  scale_y_continuous(limits = c(0,20)) + scale_x_continuous(limits = c(0,15))
    print(gg[[i]])
}

widths <- c(15, 15, 15)  # Adjust these values
heights <- c(15, 15)    # Adjust these values

patchwork::wrap_plots(plots = gg, widths = widths, heights = heights)

In [ ]:
# Create a vector of sample names to compare with sg18
samples_to_compare <- c(2, 3, 4, 5)

# Initialize an empty list to store the plots
gg <- list()

# Loop through each sample to compare with sg18
for (i in 1:length(desired_cell_types)) {
    for (sample in samples_to_compare) {
        # Create the comparison vector
        comparison_vector <- c(1, sample)
        # Generate the plots
        gg[[i]] <- netAnalysis_signalingChanges_scatter(cellchat, idents.use = desired_cell_types[i], comparison = comparison_vector)
        print(gg[[i]])
    }
}

## Part II: Identify the conserved and context-specific signaling pathways

In [ ]:
### Identify signaling groups based on their functional similarity

In [ ]:
#cellchat <- computeNetSimilarity(cellchat, type = "functional")
sub_cellchat <- computeNetSimilarityPairwise(sub_cellchat, type = "functional")

In [ ]:
netEmbedding <- function(object, slot.name = "netP", type = c("functional","structural"), comparison = NULL, pathway.remove = NULL, k = NULL) {
  if (object@options$mode == "single") {
    comparison <- "single"
    cat("Manifold learning of the signaling networks for a single dataset", '\n')
  } else if (object@options$mode == "merged") {
    if (is.null(comparison)) {
      comparison <- 1:length(unique(object@meta$datasets))
    }
    cat("Manifold learning of the signaling networks for datasets", as.character(comparison), '\n')
  }
  comparison.name <- paste(comparison, collapse = "-")
  Similarity <- methods::slot(object, slot.name)$similarity[[type]]$matrix[[comparison.name]]
  if (is.null(pathway.remove)) {
    pathway.remove <- rownames(Similarity)[which(colSums(Similarity) == 1)]
  }
  if (length(pathway.remove) > 0) {
    pathway.remove.idx <- which(rownames(Similarity) %in% pathway.remove)
    Similarity <- Similarity[-pathway.remove.idx, -pathway.remove.idx]
  }
  if (is.null(k)) {
    k <- ceiling(sqrt(dim(Similarity)[1])) + 1
  }
  options(warn = -1)
  # dimension reduction
  Y <- umap(Similarity, min_dist = 0.3, n_neighbors = k)
  if (!is.list(methods::slot(object, slot.name)$similarity[[type]]$dr)) {
    methods::slot(object, slot.name)$similarity[[type]]$dr <- NULL
  }
  methods::slot(object, slot.name)$similarity[[type]]$dr[[comparison.name]] <- Y
  return(object)
}

In [ ]:
sub_cellchat <- netEmbedding(sub_cellchat, type = "functional")
sub_cellchat <- netClustering(sub_cellchat, type = "functional")

In [ ]:
# Visualization in 2D-space
netVisual_embeddingPairwise(sub_cellchat, type = "functional", label.size = 3.5)

#### Compare the overall information flow of each signaling pathway

In [ ]:
# Create a vector of sample names to compare with sg18
samples_to_compare <- c(2, 3, 4, 5)

# Loop through each sample to compare with sg18
    for (sample in samples_to_compare) {
        # Create the comparison vector
        comparison_vector <- c(1, sample)
        # Generate the plots
        gg1 <- rankNet(cellchat, mode = "comparison", comparison = comparison_vector, stacked = T, do.stat = TRUE)
        gg2 <- rankNet(cellchat, mode = "comparison", comparison = comparison_vector, stacked = F, do.stat = TRUE)
        
        print(gg1 + gg2)
    }


In [ ]:
# Create a vector of sample names to compare with sg18
samples_to_compare <- c(1, 3, 4, 5)

# Loop through each sample to compare with sg18

    for (sample in samples_to_compare) {
        # Create the comparison vector
        comparison_vector <- c(2, sample)
        # Generate the plots
        gg1 <- rankNet(cellchat, mode = "comparison", comparison = comparison_vector, stacked = T, do.stat = TRUE)
        gg2 <- rankNet(cellchat, mode = "comparison", comparison = comparison_vector, stacked = F, do.stat = TRUE)
        
        print(gg1 + gg2)
    }

#### Compare outgoing (or incoming) signaling associated with each cell population

In [ ]:
library(ComplexHeatmap)

i = 1
# combining all the identified signaling pathways from different datasets 
pathway.union <- union(object.list[[i]]@netP$pathways, object.list[[i+1]]@netP$pathways)
ht1 = netAnalysis_signalingRole_heatmap(object.list[[i]], pattern = "outgoing", signaling = pathway.union, title = names(object.list)[i], width = 6, height =14)
ht2 = netAnalysis_signalingRole_heatmap(object.list[[i+1]], pattern = "outgoing", signaling = pathway.union, title = names(object.list)[i+1], width = 6, height = 14)
ht3 = netAnalysis_signalingRole_heatmap(object.list[[i+2]], pattern = "outgoing", signaling = pathway.union, title = names(object.list)[i+2], width = 6, height = 14)
ht4 = netAnalysis_signalingRole_heatmap(object.list[[i+3]], pattern = "outgoing", signaling = pathway.union, title = names(object.list)[i+3], width = 6, height = 14)
ht5 = netAnalysis_signalingRole_heatmap(object.list[[i+4]], pattern = "outgoing", signaling = pathway.union, title = names(object.list)[i+4], width = 6, height = 14)

draw(ht1 + ht2, ht_gap = unit(0.5, "cm"))
draw(ht3+ ht4, ht_gap = unit(0.5, "cm"))
draw(ht5)


png(file="./figures/combined_heatmap.png",width=100,height=200)
combined_heatmap <- ht1 + ht2 + ht3 + ht4 + ht5
draw(combined_heatmap)
dev.off()

In [ ]:
ht1 = netAnalysis_signalingRole_heatmap(object.list[[i]], pattern = "incoming", signaling = pathway.union, title = names(object.list)[i], width = 6, height =14, color.heatmap = "GnBu")
ht2 = netAnalysis_signalingRole_heatmap(object.list[[i+1]], pattern = "incoming", signaling = pathway.union, title = names(object.list)[i+1], width = 6, height =14, color.heatmap = "GnBu")
ht3 = netAnalysis_signalingRole_heatmap(object.list[[i+2]], pattern = "incoming", signaling = pathway.union, title = names(object.list)[i+2], width = 6, height =14, color.heatmap = "GnBu")
ht4 = netAnalysis_signalingRole_heatmap(object.list[[i+3]], pattern = "incoming", signaling = pathway.union, title = names(object.list)[i+3], width = 6, height =14, color.heatmap = "GnBu")
ht5 = netAnalysis_signalingRole_heatmap(object.list[[i+4]], pattern = "incoming", signaling = pathway.union, title = names(object.list)[i+4], width = 6, height =14, color.heatmap = "GnBu")

draw(ht1 + ht2, ht_gap = unit(0.5, "cm"))
draw(ht3+ ht4, ht_gap = unit(0.5, "cm"))
draw(ht5)


png(file="./figures/combined_heatmap.png",width=100,height=200)
combined_heatmap <- ht1 + ht2 + ht3 + ht4 + ht5
draw(combined_heatmap)
dev.off()

In [ ]:
ht1 = netAnalysis_signalingRole_heatmap(object.list[[i]], pattern = "all", signaling = pathway.union, title = names(object.list)[i], width = 6, height =14, color.heatmap = "OrRd")
ht2 = netAnalysis_signalingRole_heatmap(object.list[[i+1]], pattern = "all", signaling = pathway.union, title = names(object.list)[i+1], width = 6, height =14, color.heatmap = "OrRd")
ht3 = netAnalysis_signalingRole_heatmap(object.list[[i+2]], pattern = "all", signaling = pathway.union, title = names(object.list)[i+2], width = 6, height =14, color.heatmap = "OrRd")
ht4 = netAnalysis_signalingRole_heatmap(object.list[[i+3]], pattern = "all", signaling = pathway.union, title = names(object.list)[i+3], width = 6, height =14, color.heatmap = "OrRd")
ht5 = netAnalysis_signalingRole_heatmap(object.list[[i+4]], pattern = "all", signaling = pathway.union, title = names(object.list)[i+4], width = 6, height =14, color.heatmap = "OrRd")

draw(ht1 + ht2, ht_gap = unit(0.5, "cm"))
draw(ht3+ ht4, ht_gap = unit(0.5, "cm"))
draw(ht5)

png(file="./figures/combined_heatmap.png",width=100,height=200)
combined_heatmap <- ht1 + ht2 + ht3 + ht4 + ht5
draw(combined_heatmap)
dev.off()

## Part III: Identify the upgulated and down-regulated signaling ligand-receptor pairs

### Identify dysfunctional signaling by comparing the communication probabities

In [ ]:
# Assuming 'comparison' is a vector of dataset numbers
comparison <- c(1, 2,3, 4, 5)

# Get the dataset names from the 'sub_cellchat' object
dataset_names <- names(cellchat@images)

# Extract the dataset names corresponding to the numbers in 'comparison'
selected_dataset_names <- dataset_names[comparison]

# Print the selected dataset names
cat("Selected dataset names:", selected_dataset_names, "\n")

In [ ]:
options(repr.plot.width = 60, repr.plot.height = 15)  
netVisual_bubble(cellchat, sources.use = NULL, targets.use = NULL,  comparison = c(1, 2, 3, 4, 5), angle.x = 90)

In [ ]:
# Create a vector of sample names to compare with sg18
samples_to_compare <- c(2, 3, 4, 5)

# Loop through each sample to compare with sg18

    for (sample in samples_to_compare) {
        # Create the comparison vector
        comparison_vector <- c(1, sample)
        # Generate the plots
        options(repr.plot.width = 20, repr.plot.height = 10)  
        # identify the upgulated (increased) and down-regulated (decreased) signaling ligand-receptor pair
        net1 <- netVisual_bubble(cellchat, sources.use = NULL, targets.use = NULL,  comparison = c(1, sample), max.dataset = sample, title.name = paste("Increased signaling in", names(object.list)[sample], "(vs. sg18)"), angle.x = 90, remove.isolate = T)
        #> Comparing communications on a merged object
        net2 <- netVisual_bubble(cellchat, sources.use = NULL, targets.use = NULL,  comparison = c(1, sample), max.dataset = 1, title.name = paste("Decreased signaling in ", names(object.list)[sample], "(vs. sg18)"), angle.x = 90, remove.isolate = T)
        print(net1)
        print(net2)
    }

In [ ]:
# Create a vector of sample names to compare with sg18
samples_to_compare <- c(1, 3, 4, 5)

# Loop through each sample to compare with sg18

for (sample in samples_to_compare) {
    # Create the comparison vector
    comparison_vector <- c(2, sample)
    # Generate the plots
    options(repr.plot.width = 20, repr.plot.height = 10)  
    # identify the upgulated (increased) and down-regulated (decreased) signaling ligand-receptor pair
    net1 <- netVisual_bubble(cellchat, sources.use = NULL, targets.use = NULL,  comparison = comparison_vector, max.dataset = sample, title.name = paste("Increased signaling in", names(object.list)[sample], "(vs. sg20)"), angle.x = 90, remove.isolate = T)
    #> Comparing communications on a merged object
    net2 <- netVisual_bubble(cellchat, sources.use = NULL, targets.use = NULL,  comparison = comparison_vector, max.dataset = 2, title.name = paste("Decreased signaling in ", names(object.list)[sample], "(vs. sg20)"), angle.x = 90, remove.isolate = T)
    print(net1)
    print(net2)
}

### Identify dysfunctional signaling by using differential expression analysis

In [ ]:
# Create a vector of sample names to compare with sg18
samples_to_compare <- c(2, 3, 4, 5)

# define a positive dataset, i.e., the dataset with positive fold change against the other dataset
pos.dataset = "ctrl"
# define a char name used for storing the results of differential expression analysis
features.name = pos.dataset
# perform differential expression analysis
cellchat <- identifyOverExpressedGenes(cellchat, group.dataset = "datasets", pos.dataset = pos.dataset, features.name = features.name, only.pos = FALSE, thresh.pc = 0.1, thresh.fc = 0.1, thresh.p = 1)

# map the results of differential expression analysis onto the inferred cell-cell communications to easily manage/subset the ligand-receptor pairs of interest
net <- netMappingDEG(cellchat, features.name = features.name)
# extract the ligand-receptor pairs with upregulated ligands in sg18

for (sample in samples_to_compare) {
    # Create the comparison vector
    comparison_vector <- c(1, sample)
    # Generate the plots
    options(repr.plot.width = 20, repr.plot.height = 10)  
    
    # identify the upgulated (increased) and down-regulated (decreased) signaling ligand-receptor pair
    net.up <- subsetCommunication(cellchat, net = net, datasets ="ctrl",ligand.logFC = 0.2, receptor.logFC = NULL)
    # extract the ligand-receptor pairs with upregulated ligands and upregulated recetptors in sample2, i.e.,downregulated in sg18
    net.down <- subsetCommunication(cellchat, net = net, datasets = names(object.list)[sample],ligand.logFC = -0.1, receptor.logFC = -0.1)
    
    gene.up <- extractGeneSubsetFromPair(net.up, cellchat)
    gene.down <- extractGeneSubsetFromPair(net.down, cellchat)
    
    # We then visualize the upgulated and down-regulated signaling ligand-receptor pairs using bubble plot
    options(repr.plot.width = 20, repr.plot.height = 10)
    pairLR.use.up = net.up[, "interaction_name", drop = F]
    gg1 <- netVisual_bubble(cellchat, pairLR.use = pairLR.use.up, sources.use = NULL, targets.use = NULL, comparison = comparison_vector,  angle.x = 90, remove.isolate = T,title.name = paste0("Up-regulated signaling in ", names(object.list)[1], " vs.", names(object.list)[sample]))
    pairLR.use.down = net.down[, "interaction_name", drop = F]
    gg2 <- netVisual_bubble(cellchat, pairLR.use = pairLR.use.down, sources.use = NULL, targets.use = NULL, comparison = comparison_vector,  angle.x = 90, remove.isolate = T,title.name = paste0("Down-regulated signaling in ", names(object.list)[1], " vs.", names(object.list)[sample]))

    print(gg1)
    print(gg2)
    
    #or visualize the upgulated and down-regulated signaling ligand-receptor pairs using Chord diagram
    netVisual_chord_gene(object.list[[sample]], sources.use = NULL, targets.use = NULL, slot.name = 'net', net = net.up, lab.cex = 0.8, small.gap = 3.5, title.name = paste0("Up-regulated signaling in ", names(object.list)[sample]))
    netVisual_chord_gene(object.list[[1]], sources.use = NULL, targets.use = NULL, slot.name = 'net', net = net.down, lab.cex = 0.8, small.gap = 3.5, title.name = paste0("Down-regulated signaling in ", names(object.list)[sample]))
    
    # visualize the enriched ligands in the first condition
computeEnrichmentScore(net.down, species = 'mouse')
    # visualize the enriched ligands in the second condition
computeEnrichmentScore(net.up, species = 'mouse')
   }

In [ ]:
pathways.show <- c("COLLAGEN") 
weight.max <- getMaxWeight(object.list, slot.name = c("netP"), attribute = pathways.show) # control the edge weights across different datasets
par(mfrow = c(1,2), xpd=TRUE)
for (i in 1:length(object.list)) {
  netVisual_aggregate(object.list[[i]], signaling = pathways.show, layout = "circle", edge.weight.max = weight.max[1], edge.width.max = 10, signaling.name = paste(pathways.show, names(object.list)[i]))
}